In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=by1HNoMQrZ8tMD1TCnrBEGjm0MGbKk&access_type=offline&code_challenge=hLdEBDZY9EI78HRjZFCfI-WvwZqfr-BRmbQKzED8ifo&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning

In [4]:
import numpy as np
from typing import Any
import os
import hail as hl
import pyspark.sql.functions as f
import pandas as pd

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.susie_inf import SUSIE_inf
from gentropy.dataset.study_index import StudyIndex
from gentropy.method.window_based_clumping import WindowBasedClumping
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.carma import CARMA
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation

Loading BokehJS ...

In [5]:
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})
hl.init(sc=session.spark.sparkContext, log="/dev/null")

24/06/24 12:14:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [6]:
path_study_locus="gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer"
path_si="gs://gwas_catalog_data/study_index"
path_out="gs://genetics-portal-dev-analysis/yt4/tmp_to_delete/"

path_gwas1="gs://gwas_catalog_data/harmonised_summary_statistics/GCST90012877.parquet"
gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)

study_index = StudyIndex.from_parquet(session, path_si)
study_locus=StudyLocus.from_parquet(session, path_study_locus)
df = study_locus.df.withColumn("row_index", f.monotonically_increasing_id())
study_locus_row = df.filter(df.row_index == 3).drop("row_index").first()

In [7]:
study_locus_row

Row(studyLocusId=-7673925720032212461, variantId='7_100374211_A_G', chromosome='7', position=100374211, region=None, studyId='GCST90012877', beta=0.0896616921418, zScore=None, pValueMantissa=3.2829999923706055, pValueExponent=-18, effectAlleleFrequencyFromSource=0.6793869733810425, standardError=0.0103044894102, subStudyDescription=None, qualityControls=[], finemappingMethod=None, credibleSetIndex=None, credibleSetlog10BF=None, purityMeanR2=None, purityMinR2=None, locusStart=None, locusEnd=None, sampleSize=None, ldSet=None, locus=[Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='7_100378675_T_A', pValueMantissa=2.492000102996826, pValueExponent=-1, beta=0.376188896293, standardError=0.326478273864, r2Overall=None), Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='7_100423995_A_C', pValueMantissa=4.177000045776367, pValueExponent=-1, beta=-0.00958745201743, standardError=0.0118293041715, r2

## FUNCTION susie_finemapper_ss_gathered

In [6]:
res=SusieFineMapperStep.susie_finemapper_ss_gathered(
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
sum_pips=0.99,
)

2024-06-24 10:10:26.672 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'freq_meta' -> 'freq_meta_1'
    'age_index_dict' -> 'age_index_dict_1'
    'rf' -> 'rf_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'faf_index_dict' -> 'faf_index_dict_1'
2024-06-24 10:10:34.632 Hail: INFO: Ordering unsorted dataset with network shuffle
2024-06-24 10:10:41.448 Hail: INFO: Coerced sorted dataset
2024-06-24 10:11:47.571 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'freq_meta' -> 'freq_meta_1'
    'age_index_dict' -> 'age_index_dict_1'
    'rf' -> 'rf_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'faf_index_dict' -> 'faf_index_dict_1'
2024-06-24 10:11:53.710 Hail: INFO: Ordering unsorted dataset with net

In [7]:
print(res.df.show())

+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf|10.804377703784736|0.47131838686025107|1.168226346188087...| 8.70122609404

## FUNCTION susie_finemapper_one_sl_row_v4_ss_gathered_boundaries

In [27]:
schema = StudyLocus.get_schema()
study_locus_df = session.spark.createDataFrame([study_locus_row], schema=schema)

# Add a new column
study_locus_df = study_locus_df.withColumn("locusStart", f.col("position")-100_000)
study_locus_df = study_locus_df.withColumn("locusEnd", f.col("position")+100_000)

study_locus_df_boundaries=study_locus_df.first()

In [28]:
study_locus_df_boundaries

Row(studyLocusId=-7673925720032212461, variantId='7_100374211_A_G', chromosome='7', position=100374211, region=None, studyId='GCST90012877', beta=0.0896616921418, zScore=None, pValueMantissa=3.2829999923706055, pValueExponent=-18, effectAlleleFrequencyFromSource=0.6793869733810425, standardError=0.0103044894102, subStudyDescription=None, qualityControls=[], finemappingMethod=None, credibleSetIndex=None, credibleSetlog10BF=None, purityMeanR2=None, purityMinR2=None, locusStart=100274211, locusEnd=100474211, sampleSize=None, ldSet=None, locus=[Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='7_100378675_T_A', pValueMantissa=2.492000102996826, pValueExponent=-1, beta=0.376188896293, standardError=0.326478273864, r2Overall=None), Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='7_100423995_A_C', pValueMantissa=4.177000045776367, pValueExponent=-1, beta=-0.00958745201743, standardError=0.0118293

In [29]:
res=SusieFineMapperStep.susie_finemapper_one_sl_row_v4_ss_gathered_boundaries(
session=session,
study_locus_row=study_locus_df_boundaries,
study_index=study_index,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


2024-06-24 12:25:27.171 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'age_index_dict' -> 'age_index_dict_1'
    'rf' -> 'rf_1'
    'faf_index_dict' -> 'faf_index_dict_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'freq_meta' -> 'freq_meta_1'
    'age_distribution' -> 'age_distribution_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
2024-06-24 12:25:35.632 Hail: INFO: Ordering unsorted dataset with network shuffle
2024-06-24 12:25:42.511 Hail: INFO: Coerced sorted dataset


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf| 10.80437770378474|0.47131838686025107|1.168226346188087...| 8.70122609404

## FUNCTION susie_finemapper_one_studylocus_row_v3_dev_ss_gathered

In [8]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v3_dev_ss_gathered(
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf| 10.80437770378474|0.47131838686025107|1.168226346188087...| 8.70122609404

In [9]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v3_dev_ss_gathered(
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=True,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf| 10.80437770378474|0.47131838686025107|1.168226346188087...| 8.70122609404

In [10]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v3_dev_ss_gathered(
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=True,
run_sumstat_imputation=True,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf|10.778660326772531|0.49143982786084517|2.22734237565826E-10| 8.70122609404

In [11]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v3_dev_ss_gathered(
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=True,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf|10.778660326772531|0.49143982786084517|2.22734237565826E-10| 8.70122609404

## FUNCTION susie_finemapper_one_studylocus_row_v2_dev

In [12]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.9,
ld_score_threshold=5,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf|10.804377703784757|0.47131838686025107|1.168226346188087...| 8.70122609404

## FUNCTION susie_finemapper_one_studylocus_row

In [13]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 100000,
max_causal_snps=10,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res.df.show())


+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|         purityMinR2|            zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+-------------------+--------------------+------------------+--------------+--------------+
|               1|-7673925720032212461|GCST90012877|7:100274211-10047...|[{7_100374211_A_G...|7_100374211_A_G|         7|100374211|        SuSiE-inf|10.804377703784736|0.47131838686025107|1.168226346188087...| 8.70122609404

## BIGGER WINDOW

In [10]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 1000000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.9,
ld_score_threshold=5,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|credibleSetIndex|       studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|      purityMeanR2|       purityMinR2|          zScore|pValueMantissa|pValueExponent|
+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|               1|5658144564919822085|GCST90012877|7:99374211-101374211|[{7_100334426_C_T...|7_100334426_C_T|         7|100334426|        SuSiE-inf| 9.753310955264796|0.9402882016836946|0.8805555595382267|8.68309514557431|     2.5964453|    

In [11]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 1000000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=True,
run_sumstat_imputation=True,
carma_time_limit=6000,
imputed_r2_threshold=0.9,
ld_score_threshold=5,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|credibleSetIndex|       studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|      purityMeanR2|       purityMinR2|               z|pValueMantissa|pValueExponent|
+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|               1|5658144564919822085|GCST90012877|7:99374211-101374211|[{7_100334426_C_T...|7_100334426_C_T|         7|100334426|        SuSiE-inf| 8.297812248110523|0.8690104643462842|0.6023034424917539|8.68309514557431|     2.5964453|    

In [12]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 1000000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=True,
run_sumstat_imputation=False,
carma_time_limit=6000,
imputed_r2_threshold=0.9,
ld_score_threshold=5,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|credibleSetIndex|       studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|      purityMeanR2|       purityMinR2|               z|pValueMantissa|pValueExponent|
+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|               1|5658144564919822085|GCST90012877|7:99374211-101374211|[{7_100334426_C_T...|7_100334426_C_T|         7|100334426|        SuSiE-inf| 9.755892086737095|0.9402882016836946|0.8805555595382267|8.68309514557431|     2.5964453|    

In [13]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
GWAS=gwas1,
session=session,
study_locus_row=study_locus_row,
study_index=study_index,
radius= 1000000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=True,
carma_time_limit=6000,
imputed_r2_threshold=0.9,
ld_score_threshold=5,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|credibleSetIndex|       studyLocusId|     studyId|              region|               locus|      variantId|chromosome| position|finemappingMethod|credibleSetlog10BF|      purityMeanR2|       purityMinR2|               z|pValueMantissa|pValueExponent|
+----------------+-------------------+------------+--------------------+--------------------+---------------+----------+---------+-----------------+------------------+------------------+------------------+----------------+--------------+--------------+
|               1|5658144564919822085|GCST90012877|7:99374211-101374211|[{7_100334426_C_T...|7_100334426_C_T|         7|100334426|        SuSiE-inf| 8.297812248110523|0.8690104643462842|0.6023034424917539|8.68309514557431|     2.5964453|    

In [ ]:
#path_gwas1 = "/Users/yt4/Projects/otg_etl/input_gwas/R10_M13_LOWBACKPAIN.parquet"
#path_study_index = "/Users/yt4/Projects/otg_etl/study_index/"
#gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)
#study_index = StudyIndex.from_parquet(session, path_study_index)

path_si="gs://gwas_catalog_data/study_index"
#path_gwas1="gs://gwas_catalog_data/harmonised_summary_statistics/GCST90025949.parquet"
path_gwas1="gs://gwas_catalog_data/harmonised_summary_statistics/GCST90012877.parquet"
gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)
study_index = StudyIndex.from_parquet(session, path_si)

In [ ]:
#study_index._df.filter(f.col("pubmedId")==35361970).toPandas().to_csv("~/35361970.csv")

In [ ]:
slt=WindowBasedClumping.clump(gwas1,gwas_significance=5e-8,distance=1e6)
slt_df=slt._df
#one_row=slt_df.first()

df = slt_df.withColumn("row_index", f.monotonically_increasing_id())
one_row = df.filter(df.row_index == 18).first()

In [ ]:
slt_df.show()

In [ ]:
path_si="gs://gwas_catalog_data/study_index"
path_gwas1="gs://gwas_catalog_data/harmonised_summary_statistics/GCST90025949.parquet"
gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)
study_index = StudyIndex.from_parquet(session, path_si)


slt=WindowBasedClumping.clump(gwas1,gwas_significance=5e-8,distance=5e5)
slt_df=slt._df
#one_row=slt_df.first()

df = slt_df.withColumn("row_index", f.monotonically_increasing_id())
one_row = df.filter(df.row_index == 3).first()

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row(GWAS=gwas1,
session=session,
study_locus_row=one_row,
study_index=study_index,
radius= 100_000,
max_causal_snps = 10)
print(res._df.withColumn("size", f.size(res._df["locus"])).show())

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    radius= 100_000,
    max_causal_snps=10,
    susie_est_tausq=False,
    run_carma=False,
    run_sumstat_imputation=False,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)
sl=res["study_locus"]
print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
print(res["log"])

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    window= 100_000,
    L=10,
    susie_est_tausq=False,
    run_carma=False,
    run_sumstat_imputation=False,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)
sl=res["study_locus"]
print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
print(res["log"])

In [ ]:
cred_sets

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    window= 1_000_000,
    L=10,
    susie_est_tausq=False,
    run_carma=True,
    run_sumstat_imputation=True,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)
sl=res["study_locus"]
print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
print(res["log"])

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    window= 1_000_000,
    L=10,
    susie_est_tausq=False,
    run_carma=True,
    run_sumstat_imputation=False,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)
sl=res["study_locus"]
print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
print(res["log"])

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    window= 1_000_000,
    L=10,
    susie_est_tausq=False,
    run_carma=False,
    run_sumstat_imputation=False,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)
sl=res["study_locus"]
print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
print(res["log"])

In [ ]:
for i in range(0,min(3,df.count())):
    one_row = df.filter(df.row_index == i).first()

    res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
        GWAS=gwas1,
        session=session,
        study_locus_row=one_row,
        study_index=study_index,
        window= 1_000_000,
        L=10,
        susie_est_tausq=False,
        run_carma=True,
        run_sumstat_imputation=True,
        carma_time_limit=600,
        imputed_r2_threshold=0.8,
        ld_score_threshold=4
    )

    sl=res["study_locus"]
    print(sl._df.withColumn("size", f.size(sl._df["locus"])).show())
    print(res["log"])

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row(GWAS=gwas1,
session=session,
study_locus_row=one_row,
study_index=study_index,
window= 1_000_000,
L = 10)

In [ ]:
res._df.withColumn("size", f.size(res._df["locus"])).show()

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v2_dev(
    GWAS=gwas1,
    session=session,
    study_locus_row=one_row,
    study_index=study_index,
    window= 1_000_000,
    L=10,
    susie_est_tausq=False,
    run_carma=True,
    run_sumstat_imputation=True,
    carma_time_limit=600,
    imputed_r2_threshold=0.8,
    ld_score_threshold=4
)

In [ ]:
sl=res["study_locus"]
sl._df.withColumn("size", f.size(sl._df["locus"])).show()

In [ ]:
print(res["log"])

# Inside fucntion

In [ ]:
GWAS=gwas1
session=session
study_locus_row=one_row
study_index=study_index
window= 100_000
L=10
susie_est_tausq=False
run_carma=True
run_sumstat_imputation=True
carma_time_limit=100
imputed_r2_threshold=0.8
ld_score_threshold=4
secondary_signal_pval_threshold=1e-7
primary_signal_pval_threshold=5e-8
purity_min_r2_threshold = 0.25

In [ ]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

chromosome = study_locus_row["chromosome"]
position = study_locus_row["position"]
studyId = study_locus_row["studyId"]

study_index_df = study_index._df
study_index_df = study_index_df.filter(f.col("studyId") == studyId)
major_population = study_index_df.select(
    "studyId",
    f.array_max(f.col("ldPopulationStructure"))
    .getItem("ldPopulation")
    .alias("majorPopulation"),
).collect()[0]["majorPopulation"]

region = (
    chromosome
    + ":"
    + str(int(position - window / 2))
    + "-"
    + str(int(position + window / 2))
)

gwas_df = (
    GWAS.df.withColumn("z", f.col("beta") / f.col("standardError"))
    .withColumn(
        "chromosome", f.split(f.col("variantId"), "_")[0].cast("string")
    )
    .withColumn("position", f.split(f.col("variantId"), "_")[1].cast("int"))
    .filter(f.col("studyId") == studyId)
    .filter(f.col("z").isNotNull())
    .filter(f.col("chromosome") == chromosome)
    .filter(f.col("position") >= position - window / 2)
    .filter(f.col("position") <= position + window / 2)
)

ld_index = (
    GnomADLDMatrix()
    .get_locus_index(
        study_locus_row=study_locus_row,
        radius=window,
        major_population=major_population,
    )
    .withColumn(
        "variantId",
        f.concat(
            f.lit(chromosome),
            f.lit("_"),
            f.col("`locus.position`"),
            f.lit("_"),
            f.col("alleles").getItem(0),
            f.lit("_"),
            f.col("alleles").getItem(1),
        ).cast("string"),
    )
)

gnomad_ld = GnomADLDMatrix.get_numpy_matrix(
    ld_index, gnomad_ancestry=major_population
)

In [ ]:
GWAS_df = gwas_df.toPandas()
import time

pd.DataFrame.iteritems = pd.DataFrame.items

start_time = time.time()
#GWAS_df = GWAS_df.toPandas()
ld_index = ld_index.toPandas()
ld_index = ld_index.reset_index()

N_gwas = len(GWAS_df)
N_ld = len(ld_index)

# Filtering out the variants that are not in the LD matrix, we don't need them
df_columns = ["variantId", "z"]
GWAS_df = GWAS_df.merge(ld_index, on="variantId", how="inner")
GWAS_df = GWAS_df[df_columns].reset_index()
N_after_merge = len(GWAS_df)

merged_df = GWAS_df.merge(
    ld_index, left_on="variantId", right_on="variantId", how="inner"
)
indices = merged_df["index_y"].values

ld_to_fm = gnomad_ld[indices][:, indices]
z_to_fm = GWAS_df["z"].values

In [ ]:
susie_output = SUSIE_inf.susie_inf(
    z=z_to_fm, LD=ld_to_fm, L=L, est_tausq=susie_est_tausq
)

In [ ]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, DoubleType        
schema = StructType([
    StructField("variantId", StringType(), True),
    StructField("z", DoubleType(), True)  # assuming 'z' is of type DoubleType
])
variant_index = (
    session.spark.createDataFrame(
        GWAS_df[["variantId", "z"]],
        schema=schema,
    )
    .withColumn(
        "chromosome", f.split(f.col("variantId"), "_")[0].cast("string")
    )
    .withColumn("position", f.split(f.col("variantId"), "_")[1].cast("int"))
)

In [ ]:
ld_matrix=ld_to_fm
cs_lbf_thr = 2
sum_pips = 0.99


from __future__ import annotations

import time
from typing import Any

import hail as hl
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import scipy as sc
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from gentropy.common.session import Session
from gentropy.common.spark_helpers import neglog_pvalue_to_mantissa_and_exponent
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.carma import CARMA
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation
from gentropy.method.susie_inf import SUSIE_inf

In [ ]:
variants = np.array(
    [row["variantId"] for row in variant_index.select("variantId").collect()]
).reshape(-1, 1)

PIPs = susie_output["PIP"]
lbfs = susie_output["lbf_variable"]
mu = susie_output["mu"]
susie_result = np.hstack((variants, PIPs, lbfs, mu))

L_snps = PIPs.shape[1]

# Extracting credible sets
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)
cred_sets = None
counter = 0
for i, cs_lbf_value in order_creds:
    if counter > 0 and cs_lbf_value < cs_lbf_thr:
        counter += 1
        continue
    counter += 1
    sorted_arr = susie_result[
        susie_result[:, i + 1].astype(float).argsort()[::-1]
    ]
    cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
    filter_row = np.argmax(cumsum_arr >= sum_pips)
    if filter_row == 0 and cumsum_arr[0] < sum_pips:
        filter_row = len(cumsum_arr)
    filter_row += 1
    filtered_arr = sorted_arr[:filter_row]
    cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
    win = Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )

    cred_set = (
        session.spark.createDataFrame(
            cred_set.tolist(),
            ["variantId", "posteriorProbability", "logBF", "beta"],
        )
        .join(
            variant_index.select(
                "variantId",
                "chromosome",
                "position",
            ),
            "variantId",
        )
        .sort(f.desc("posteriorProbability"))
        .withColumn(
            "locus",
            f.collect_list(
                f.struct(
                    f.col("variantId").cast("string").alias("variantId"),
                    f.col("posteriorProbability")
                    .cast("double")
                    .alias("posteriorProbability"),
                    f.col("logBF").cast("double").alias("logBF"),
                    f.col("beta").cast("double").alias("beta"),
                )
            ).over(win),
        )
        .limit(1)
        .withColumns(
            {
                "studyId": f.lit(studyId),
                "region": f.lit(region),
                "credibleSetIndex": f.lit(counter),
                "credibleSetlog10BF": f.lit(cs_lbf_value * 0.4342944819),
                "finemappingMethod": f.lit("SuSiE-inf"),
            }
        )
        .withColumn(
            "studyLocusId",
            StudyLocus.assign_study_locus_id(
                f.col("studyId"), f.col("variantId")
            ),
        )
        .select(
            "studyLocusId",
            "studyId",
            "region",
            "credibleSetIndex",
            "locus",
            "variantId",
            "chromosome",
            "position",
            "finemappingMethod",
            "credibleSetlog10BF",
        )
    )
    if cred_sets is None:
        cred_sets = cred_set
    else:
        cred_sets = cred_sets.unionByName(cred_set)


In [ ]:
cred_sets.show()

In [ ]:
import scipy as sc

# Calulating purity
variant_index_df = variant_index.toPandas()
cred_sets_variantId = cred_sets.select("locus.variantId").toPandas()

lead_variantId_list = (
    cred_sets.select("variantId").toPandas()["variantId"].tolist()
)
cred_set_index=cred_sets.select("credibleSetIndex").toPandas()["credibleSetIndex"].tolist()
vlist_series = pd.Series(lead_variantId_list)
ind = vlist_series.map(variant_index_df.set_index("variantId").index.get_loc)
z_values = variant_index_df.iloc[ind]["z"].tolist()
z_values_array = np.array(z_values)
pval = sc.stats.chi2.sf((z_values_array**2), 1)
pval[pval < 1e-322] = 1e-322
neglogpval = -np.log10(pval)
neglogpval = neglogpval.tolist()

list_purity_mean_r2 = []
list_purity_min_r2 = []
for _, row in cred_sets_variantId.iterrows():
    row = row.iloc[0]
    vlist_series = pd.Series(row)
    ind = vlist_series.map(
        variant_index_df.set_index("variantId").index.get_loc
    )
    # print(variant_index_df.iloc[ind,0]==vlist)
    squared_matrix = ld_matrix[ind, :][:, ind] ** 2
    purity_mean_r2 = np.mean(squared_matrix)
    purity_min_r2 = np.min(squared_matrix)
    list_purity_mean_r2.append(purity_mean_r2)
    list_purity_min_r2.append(purity_min_r2)

cred_sets = cred_sets.drop("pValueMantissa", "pValueExponent")

df = pd.DataFrame(
    {
        "credibleSetIndex": cred_set_index,
        "purity_mean_r2": purity_mean_r2,
        "purity_min_r2": purity_min_r2,
        "z": z_values,
        "neglogpval": neglogpval,
    }
)
schema = StructType(
    [
        StructField("credibleSetIndex", IntegerType(), True),
        StructField("purity_mean_r2", DoubleType(), True),
        StructField("purity_min_r2", DoubleType(), True),
        StructField("z", DoubleType(), True),
        StructField("neglogpval", DoubleType(), True),
    ]
)

df_spark = session.spark.createDataFrame(df, schema=schema)


In [ ]:
df_spark.show()

In [ ]:
cred_sets.show()

In [ ]:
df_spark = session.spark.createDataFrame(df, schema=schema)

cred_sets = cred_sets.join(df_spark, on="credibleSetIndex")

mantissa, exponent = neglog_pvalue_to_mantissa_and_exponent(
    cred_sets.neglogpval
)

cred_sets = cred_sets.withColumn("pValueMantissa", mantissa)
cred_sets = cred_sets.withColumn("pValueExponent", exponent)

cred_sets = cred_sets.withColumn(
    "pValueMantissa", f.col("pValueMantissa").cast("float")
)

cred_sets = cred_sets.filter(
    (f.col("neglogpval") >= -np.log10(secondary_signal_pval_threshold))
    | (f.col("credibleSetIndex") == 1)
)

cred_sets = cred_sets.filter(
    (f.col("neglogpval") >= -np.log10(primary_signal_pval_threshold))
    | (f.col("credibleSetIndex") > 1)
)

cred_sets = cred_sets.drop("neglogpval")

cred_sets = cred_sets.filter(f.col("purity_min_r2") >= purity_min_r2_threshold)

window = Window.partitionBy("studyLocusId").orderBy("credibleSetIndex")
cred_sets = cred_sets.withColumn("rank", row_number().over(window))
cred_sets = cred_sets.filter(cred_sets["rank"] == 1).drop("rank")

In [ ]:
cred_sets.show()

In [ ]:
StudyLocus(
            _df=cred_sets,
            _schema=StudyLocus.get_schema(),
        )

In [ ]:
SL=SusieFineMapperStep.susie_inf_to_studylocus(susie_output=susie_output,
                                            session=session,
                                            studyId=studyId,
                                            region=region,
                                            variant_index=variant_index
                                            )

In [ ]:
cred_sets=SL.df
cred_sets.show(1,False,True)

In [ ]:
cred_sets_variantId = cred_sets.select("locus.variantId").toPandas()

In [ ]:
cred_sets_variantId

In [ ]:
variant_index_df=variant_index.toPandas()

In [ ]:
vlist=cred_sets_variantId.iloc[0,0]

In [ ]:
vlist

In [ ]:
vlist_series = pd.Series(vlist)
ind = vlist_series.map(variant_index_df.set_index('variantId').index.get_loc)

In [ ]:
variant_index_df.iloc[ind,0]==vlist

In [ ]:
for index, row in cred_sets_variantId.iterrows():
    row=row.iloc[0]
    vlist_series = pd.Series(row)
    ind = vlist_series.map(variant_index_df.set_index('variantId').index.get_loc)
    print(variant_index_df.iloc[ind,0]==vlist)
    cred_sets = cred_sets.withColumn(
    "purity_mean_r2",
    when((cred_sets["index"] == index_to_replace), value_to_assign).otherwise(cred_sets["purity_mean_r2"])
)

In [ ]:
squared_matrix = ld_to_fm[[23,45],:][:,[23,45]] ** 2
purity_mean_r2 = np.mean(squared_matrix)

In [ ]:
cred_sets

In [ ]:
purity_mean_r2

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, when

from pyspark.sql.functions import lit

cred_sets = cred_sets.withColumn("purity_mean_r2", lit(None).cast("string"))

# Add an 'index' column
cred_sets = cred_sets.withColumn("index", monotonically_increasing_id())

value_to_assign = 0.5  # replace with the value you want to assign
index_to_replace = 3  # replace with the index of the row you want to replace

cred_sets = cred_sets.withColumn(
    "purity_mean_r2",
    when((cred_sets["index"] == index_to_replace), value_to_assign).otherwise(cred_sets["purity_mean_r2"])
)

In [ ]:
carma_output = CARMA.time_limited_CARMA_spike_slab_noEM(
    z=z_to_fm, ld=ld_to_fm, sec_threshold=carma_time_limit
)
if carma_output["Outliers"] != [] and carma_output["Outliers"] is not None:
    GWAS_df.drop(carma_output["Outliers"], inplace=True)
    GWAS_df = GWAS_df.reset_index()
    ld_index = ld_index.reset_index()
    merged_df = GWAS_df.merge(
        ld_index, left_on="variantId", right_on="variantId", how="inner"
    )
    indices = merged_df["index_y"].values

    ld_to_fm = gnomad_ld[indices][:, indices]
    z_to_fm = GWAS_df["z"].values
    N_outliers = len(carma_output["Outliers"])
else:
    N_outliers = 0

# CARMA testing

In [ ]:
z=z_to_fm
ld=ld_to_fm
lambda_val=1
Max_Model_Dim=200000
all_iter=1
all_inner_iter=10
epsilon_threshold=1e-5
num_causal=10
tau=0.04
outlier_switch=True
outlier_BF_index = 1 / 3.2

In [ ]:
p_snp = len(z)
epsilon_list = epsilon_threshold * p_snp
all_epsilon_threshold = epsilon_threshold * p_snp

In [ ]:
z=z
ld_matrix=ld
epsilon=epsilon_list
Max_Model_Dim=Max_Model_Dim
lambda_val=lambda_val
outlier_switch=outlier_switch
tau=tau
num_causal=num_causal
inner_all_iter=all_inner_iter
outlier_BF_index=outlier_BF_index

input_conditional_S_list= None
inner_all_iter=10

In [ ]:
p = len(z)
marginal_likelihood = CARMA._ind_Normal_fixed_sigma_marginal_external
tau_sample = tau
if outlier_switch:
    outlier_likelihood = CARMA._outlier_ind_Normal_marginal_external
    outlier_tau = tau

B = Max_Model_Dim
stored_bf = 0
Sigma = ld_matrix

S = []

null_model = ""
null_margin = CARMA._prior_dist(null_model, lambda_val=lambda_val, p=p)

B_list = pd.DataFrame({"set_gamma_margin": [null_margin], "matrix_gamma": [""]})

if input_conditional_S_list is None:
    conditional_S = []
else:
    conditional_S = input_conditional_S_list
    S = conditional_S


In [ ]:
import concurrent.futures
from itertools import combinations
from math import floor, lgamma
from typing import Any

import numpy as np
import pandas as pd
from scipy.linalg import det, inv, pinv
from scipy.optimize import minimize_scalar

In [ ]:
_i=1
for _j in range(0, 10):
    set_gamma = CARMA._set_gamma_func(
        input_S=S, p=p, condition_index=conditional_S
    )

    if conditional_S is None:
        working_S = S
    else:
        working_S = np.sort(np.setdiff1d(S, conditional_S)).astype(int)

    set_gamma_margin: list[Any] = [None, None, None]
    set_gamma_prior: list[Any] = [None, None, None]
    matrix_gamma: list[Any] = [None, None, None]

    for i in range(0, len(set_gamma)):
        if set_gamma[i] is not None:
            matrix_gamma[i] = CARMA._index_fun(set_gamma[i])
            p_S = set_gamma[i].shape[1]
            set_gamma_margin[i] = np.apply_along_axis(
                marginal_likelihood,
                1,
                set_gamma[i] + 1,
                Sigma=Sigma,
                z=z,
                tau=tau_sample,
                p_S=p_S,
            )
            set_gamma_prior[i] = np.array(
                [
                    CARMA._prior_dist(model, lambda_val=lambda_val, p=p)
                    for model in matrix_gamma[i]
                ]
            )
            set_gamma_margin[i] = set_gamma_prior[i] + set_gamma_margin[i]
        else:
            set_gamma_margin[i] = np.array(null_margin)
            set_gamma_prior[i] = 0
            matrix_gamma[i] = np.array(null_model)

    columns = ["set_gamma_margin", "matrix_gamma"]
    add_B = pd.DataFrame(columns=columns)

    for i in range(len(set_gamma)):
        if isinstance(set_gamma_margin[i].tolist(), list):
            new_row = pd.DataFrame(
                {
                    "set_gamma_margin": set_gamma_margin[i].tolist(),
                    "matrix_gamma": matrix_gamma[i].tolist(),
                }
            )
            add_B = pd.concat([add_B, new_row], ignore_index=True)
        else:
            new_row = pd.DataFrame(
                {
                    "set_gamma_margin": [set_gamma_margin[i].tolist()],
                    "matrix_gamma": [matrix_gamma[i].tolist()],
                }
            )
            add_B = pd.concat([add_B, new_row], ignore_index=True)

    # Add visited models into the storage space of models
    B_list = pd.concat([B_list, add_B], ignore_index=True)
    B_list = B_list.drop_duplicates(
        subset="matrix_gamma", ignore_index=True
    )
    B_list = B_list.sort_values(
        by="set_gamma_margin", ignore_index=True, ascending=False
    )

    if len(working_S) == 0:
        # Create a DataFrame set.star
        set_star = pd.DataFrame(
            {
                "set_index": [0, 1, 2],
                "gamma_set_index": [np.nan, np.nan, np.nan],
                "margin": [np.nan, np.nan, np.nan],
            }
        )

        # Assuming set.gamma.margin and current.log.margin are defined
        aa = set_gamma_margin[1]
        aa = aa - aa[np.argmax(aa)]

        min_half_len = min(len(aa), floor(p / 2))
        decr_ind = np.argsort(np.exp(aa))[::-1]
        decr_half_ind = decr_ind[:min_half_len]

        probs = np.exp(aa)[decr_half_ind]

        chosen_index = np.random.choice(
            decr_half_ind, 1, p=probs / np.sum(probs)
        )
        set_star.at[1, "gamma_set_index"] = chosen_index[0]
        set_star.at[1, "margin"] = set_gamma_margin[1][chosen_index[0]]

        S = set_gamma[1][chosen_index[0]].tolist()

    else:
        set_star = pd.DataFrame(
            {
                "set_index": [0, 1, 2],
                "gamma_set_index": [np.nan, np.nan, np.nan],
                "margin": [np.nan, np.nan, np.nan],
            }
        )
        for i in range(0, 3):
            aa = set_gamma_margin[i]
            if np.size(aa) > 1:
                aa = aa - aa[np.argmax(aa)]
                chosen_index = np.random.choice(
                    range(0, np.size(set_gamma_margin[i])),
                    1,
                    p=np.exp(aa) / np.sum(np.exp(aa)),
                )
                set_star.at[i, "gamma_set_index"] = chosen_index
                set_star.at[i, "margin"] = set_gamma_margin[i][chosen_index]
            else:
                set_star.at[i, "gamma_set_index"] = 0
                set_star.at[i, "margin"] = set_gamma_margin[i]

        if outlier_switch:
            for i in range(1, len(set_gamma)):
                test_log_BF: float = 100
                while True:
                    aa = set_gamma_margin[i]
                    aa = aa - aa[np.argmax(aa)]
                    chosen_index = np.random.choice(
                        range(0, np.size(set_gamma_margin[i])),
                        1,
                        p=np.exp(aa) / np.sum(np.exp(aa)),
                    )
                    set_star.at[i, "gamma_set_index"] = chosen_index
                    set_star.at[i, "margin"] = set_gamma_margin[i][
                        chosen_index
                    ]

                    test_S = set_gamma[i][int(chosen_index), :]

                    modi_Sigma = Sigma.copy()
                    if np.size(test_S) > 1:
                        modi_ld_S = modi_Sigma[test_S][:, test_S]

                        result = minimize_scalar(
                            CARMA._ridge_fun,
                            bounds=(0, 1),
                            args=(
                                Sigma,
                                modi_ld_S,
                                test_S,
                                z,
                                outlier_tau,
                                outlier_likelihood,
                            ),
                            method="bounded",
                        )
                        modi_ld_S = result.x * modi_ld_S + (
                            1 - result.x
                        ) * np.eye(len(modi_ld_S))

                        modi_Sigma[np.ix_(test_S, test_S)] = modi_ld_S

                        test_log_BF = outlier_likelihood(
                            test_S + 1, Sigma, z, outlier_tau, len(test_S)
                        ) - outlier_likelihood(
                            test_S + 1,
                            modi_Sigma,
                            z,
                            outlier_tau,
                            len(test_S),
                        )
                        test_log_BF = -np.abs(test_log_BF)

                    if np.exp(test_log_BF) < outlier_BF_index:
                        set_gamma[i] = np.delete(
                            set_gamma[i],
                            int(set_star["gamma_set_index"][i]),
                            axis=0,
                        )
                        set_gamma_margin[i] = np.delete(
                            set_gamma_margin[i],
                            int(set_star["gamma_set_index"][i]),
                            axis=0,
                        )
                        conditional_S = np.concatenate(
                            [conditional_S, np.setdiff1d(test_S, working_S)]
                        )
                        conditional_S = (
                            np.unique(conditional_S).astype(int).tolist()
                        )
                    else:
                        break

        if len(working_S) == num_causal:
            set_star = set_star.drop(1)
            aa = set_star["margin"] - max(set_star["margin"])
            sec_sample = np.random.choice(
                [0, 2], 1, p=np.exp(aa) / np.sum(np.exp(aa))
            )
            ind_sec = int(
                set_star["gamma_set_index"][
                    set_star["set_index"] == int(sec_sample)
                ]
            )
            S = set_gamma[sec_sample[0]][ind_sec].tolist()
        else:
            aa = set_star["margin"] - max(set_star["margin"])
            sec_sample = np.random.choice(
                range(0, 3), 1, p=np.exp(aa) / np.sum(np.exp(aa))
            )
            if set_gamma[sec_sample[0]] is not None:
                S = set_gamma[sec_sample[0]][
                    int(set_star["gamma_set_index"][sec_sample[0]])
                ].tolist()
            else:
                S = []

    for item in conditional_S:
        if item not in S:
            S.append(item)
# END h_ind loop


In [ ]:
ld_to_fm = gnomad_ld[index_to_fm][:, index_to_fm]
ld_to_fm.shape

In [ ]:
#
if conditional_S is not None:
    all_c_index = []
    index_array = [s.split(",") for s in B_list["matrix_gamma"]]
    for tt in conditional_S:
        tt_str = str(tt)
        ind = [
            i for i, sublist in enumerate(index_array) if tt_str in sublist
        ]
        all_c_index.extend(ind)

    all_c_index = list(set(all_c_index))

    if len(all_c_index) > 0:
        temp_B_list = B_list.copy()
        temp_B_list = B_list.drop(all_c_index)
    else:
        temp_B_list = B_list.copy()
else:
    temp_B_list = B_list.copy()

result_B_list = temp_B_list[: min(int(B), len(temp_B_list))]

rb1 = result_B_list["set_gamma_margin"]

difference = abs(rb1[: (len(rb1) // 4)].mean() - stored_bf)

if difference < epsilon:
    pass
else:
    stored_bf = rb1[: (len(rb1) // 4)].mean()

In [ ]:
S

In [ ]:
difference

In [ ]:
set_gamma

In [ ]:
set_gamma

In [ ]:
sec_sample

In [ ]:
set_star

In [ ]:
sec_sample = np.random.choice(
    range(0, 3), 1, p=np.exp(aa) / np.sum(np.exp(aa))
)

In [ ]:
sec_sample=[0]

In [ ]:
int(set_star["gamma_set_index"][sec_sample[0]])

In [ ]:
set_star

In [ ]:
set_gamma[sec_sample[0]][
                int(set_star["gamma_set_index"][sec_sample[0]])
            ].tolist()

In [ ]:
set_gamma

In [ ]:
if set_gamma[sec_sample[0]] is not None:
    S = set_gamma[sec_sample[0]][
        int(set_star["gamma_set_index"][sec_sample[0]])
    ].tolist()

In [ ]:
set_gamma[sec_sample[0]][
        int(set_star["gamma_set_index"][sec_sample[0]])
    ].tolist()

In [ ]:
CARMA._set_gamma_func(
                    input_S=[1], p=p, condition_index=conditional_S
                )

In [ ]:
set_gamma_margin

In [ ]:
set_star

In [ ]:
S=[]
for item in conditional_S:
    if item not in S:
        S.append(item)

In [ ]:
S

In [ ]:
conditional_S

In [ ]:
S=[0,1,2]

In [ ]:
np.array(S)[[1,2]]